In [10]:
import pandas as pd
df = pd.read_csv(r'C:\Users\HP\Desktop\ML\Machine Learning\Projects\AI_ML_Engineer\Days to the death of a banana(computer vision regression)\data\banana_days_471_synced.csv') 

df

,image_filename,days_left
0,banana_01_day_01.jpg,8
1,banana_02_day_01.jpg,8
2,banana_03_day_01.jpg,8
3,banana_04_day_01.jpg,8
4,banana_05_day_01.jpg,8
...,...,...
466,banana_60_day_06.jpg,3
467,banana_61_day_06.jpg,3
468,banana_55_day_04.jpg,5
469,banana_56_day_03.jpg,6


In [2]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# 1. Setup Data Generators (with Augmentation)
# We use flow_from_dataframe since we have a CSV of regression labels
datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize pixel values
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2        # Use 20% of data for validation
)

# Load CSV
df = pd.read_csv(r'C:\Users\HP\Desktop\ML\Machine Learning\Projects\AI_ML_Engineer\Days to the death of a banana(computer vision regression)\data\banana_days_471_synced.csv') # Columns: ['filename', 'days_left']
df['days_left'] = df['days_left'].astype(float) # Ensure regression targets are floats

# Training Data
train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=r'C:\Users\HP\Desktop\ML\Machine Learning\Projects\AI_ML_Engineer\Days to the death of a banana(computer vision regression)\data\banana_images_jpg',
    x_col='image_filename',
    y_col='days_left',
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw', # 'raw' is essential for regression
    subset='training'
)

# Validation Data
val_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=r'C:\Users\HP\Desktop\ML\Machine Learning\Projects\AI_ML_Engineer\Days to the death of a banana(computer vision regression)\data\banana_images_jpg',
    x_col='image_filename',
    y_col='days_left',
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    subset='validation'
)

# 2. Build the Model (Transfer Learning)
def build_model():
    # Load MobileNetV2 without the top classification layer
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Freeze the base model layers
    base_model.trainable = False

    # Add custom Regression Head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)  # Prevent overfitting
    x = Dense(128, activation='relu')(x)
    predictions = Dense(1, activation='linear')(x) # Linear activation for regression

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

model = build_model()

# 3. Compile
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# 4. Train
print("Starting training...")
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator
)

# # 5. Fine-Tuning (Optional but recommended)
# # Unfreeze the last few layers of MobileNet for better accuracy
# base_model = model.layers[0]
# base_model.trainable = True
# # Fine-tune only the top 20 layers
# for layer in base_model.layers[:-20]:
#     layer.trainable = False

model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
model.fit(train_generator, epochs=10, validation_data=val_generator)

# 6. Save for Production
# Saving in .keras format is the modern standard for TF 2.x
model.save('banana_model.keras')
print("Model saved as banana_model.keras")

Found 377 validated image filenames.
Found 94 validated image filenames.
Starting training...


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 13s 817ms/step - loss: 6.9692 - mae: 2.1431 - val_loss: 17.4614 - val_mae: 4.0091
Epoch 2/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 728ms/step - loss: 1.4548 - mae: 0.9566 - val_loss: 10.0949 - val_mae: 2.8563
Epoch 3/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 718ms/step - loss: 0.9455 - mae: 0.7822 - val_loss: 5.0604 - val_mae: 1.9055
Epoch 4/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 721ms/step - loss: 0.7558 - mae: 0.6980 - val_loss: 7.9621 - val_mae: 2.4628
Epoch 5/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 668ms/step - loss: 0.7800 - mae: 0.7212 - val_loss: 7.9237 - val_mae: 2.4795
Epoch 6/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 679ms/step - loss: 0.6697 - mae: 0.6524 - val_loss: 6.2837 - val_mae: 2.1592
Epoch 7/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 670ms/step - loss: 0.5684 - mae: 0.6029 - val_loss: 5.9417 - val_mae: 2.0998
Epoch 8/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 683ms/step - loss: 0.6239 - mae: 0.6190 - val_loss: 7.2282 - val_mae: 2.3474
Epoch 9/15
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 669ms/step -

In [3]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import tensorflow as tf
import numpy as np
from PIL import Image
import io

app = FastAPI(title="Banana Death Predictor (TF)")

# CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global Model Variable
model = None

@app.on_event("startup")
def load_model():
    global model
    # Load the model once on startup
    try:
        model = tf.keras.models.load_model('banana_model.keras')
        print("TensorFlow Model Loaded Successfully")
    except Exception as e:
        print(f"Error loading model: {e}")

def preprocess_image(image_bytes):
    """
    Convert bytes to a format compatible with MobileNetV2 training
    (224x224, RGB, normalized 0-1)
    """
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    image = image.resize((224, 224))
    image_array = np.array(image)
    
    # Normalize pixel values to [0, 1] as we did in training (rescale=1./255)
    image_array = image_array / 255.0
    
    # Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    image_array = np.expand_dims(image_array, axis=0)
    return image_array

@app.get("/")
def home():
    return {"status": "Banana TensorFlow System Operational"}

@app.post("/predict")
async def predict_banana(file: UploadFile = File(...)):
    if not model:
        return {"error": "Model not loaded"}

    image_bytes = await file.read()
    processed_image = preprocess_image(image_bytes)
    
    # Predict
    prediction = model.predict(processed_image)
    
    # Extract float value
    days_left = float(prediction[0][0])
    
    # Logical capping
    days_left = max(0, days_left)

    return {
        "days_left": round(days_left, 1),
        "message": get_banana_message(days_left)
    }

def get_banana_message(days):
    if days > 7: return "Green and mean."
    if days > 3: return "Perfect eating window."
    if days > 1: return "EAT ME NOW."
    return "Banana Bread Time."

C:\Users\HP\AppData\Local\Temp\ipykernel_10192\1653531681.py:21: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
